# Fabric Insurance Demo — Prerequisite Deployment

**Purpose**: Single-notebook deployment of all Fabric artifacts for the Insurance Demo.

**What this notebook does**:
1. Creates 3 Lakehouses (Bronze, Silver, Gold)
2. Creates 1 Warehouse
3. Uploads all 13 pipeline notebooks
4. Optionally executes all notebooks in order (Bronze → Silver → Gold)

**Mock data** is generated by `nb_00_generate_mock_data` (writes CSVs directly to Bronze Lakehouse).

**Prerequisites**:
- Run this notebook **inside your target Fabric workspace**
- Workspace must have Fabric capacity assigned\n
- User must have **Contributor** or **Admin** role on the workspace

**Runtime**: ~2 minutes for deployment, ~15-25 minutes if executing all notebooks

## 0. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Update these values for your environment
# ============================================================

WORKSPACE_ID = "<YOUR_WORKSPACE_ID>"  # Find via: https://app.fabric.microsoft.com → Workspace URL

# Lakehouse names (per constitution naming convention)
LAKEHOUSES = [
    "lh_bronze_insurance",
    "lh_silver_insurance",
    "lh_gold_insurance"
]

# Warehouse name
WAREHOUSE_NAME = "wh_insurance"

# Notebook execution order (Bronze → Silver → Gold)
NOTEBOOK_EXECUTION_ORDER = [
    "nb_00_generate_mock_data",
    "nb_01_bronze_ingest",
    "nb_02_silver_customers",
    "nb_03_silver_agents",
    "nb_04_silver_policies",
    "nb_05_silver_coverages",
    "nb_06_silver_premiums",
    "nb_07_silver_claims",
    "nb_08_silver_claim_payments",
    "nb_09_gold_claims_summary",
    "nb_10_gold_premium_revenue",
    "nb_11_gold_customer_360",
    "nb_12_gold_kpi_metrics"
]

# Set to True to run all notebooks after deployment
EXECUTE_AFTER_DEPLOY = False

print("Configuration loaded")
print(f"   Workspace: {WORKSPACE_ID}")
print(f"   Lakehouses: {len(LAKEHOUSES)}")
print(f"   Notebooks to deploy: {len(NOTEBOOK_EXECUTION_ORDER)}")
print(f"   Execute after deploy: {EXECUTE_AFTER_DEPLOY}")

## 1. Authentication & Helper Functions

In [ ]:
import requests
import json
import time
import base64
import os

# --- Authenticate using Fabric's built-in token provider ---
from notebookutils import mssparkutils

def get_fabric_token():
    """Get bearer token for Fabric REST API."""
    return mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

def get_onelake_token():
    """Get bearer token for OneLake DFS API."""
    return mssparkutils.credentials.getToken("https://storage.azure.com")

def get_auth_headers():
    return {
        "Authorization": f"Bearer {get_fabric_token()}",
        "Content-Type": "application/json"
    }

FABRIC_API = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}"

def fabric_post(endpoint, body, ignore_conflict=True):
    """POST to Fabric REST API with optional 409 conflict handling."""
    resp = requests.post(
        f"{FABRIC_API}/{endpoint}",
        headers=get_auth_headers(),
        json=body
    )
    if resp.status_code == 409 and ignore_conflict:
        return {"status": "already_exists", "code": 409}
    elif resp.status_code in [200, 201]:
        return resp.json() if resp.text else {"status": "created"}
    elif resp.status_code == 202:
        return {"status": "accepted", "location": resp.headers.get("Location", "")}
    else:
        print(f"   Error {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

def fabric_get(endpoint):
    """GET from Fabric REST API."""
    resp = requests.get(
        f"{FABRIC_API}/{endpoint}",
        headers=get_auth_headers()
    )
    resp.raise_for_status()
    return resp.json()

def wait_for_operation(location_url, timeout=300):
    """Poll a long-running operation until completion."""
    start = time.time()
    while time.time() - start < timeout:
        resp = requests.get(location_url, headers=get_auth_headers())
        if resp.status_code == 200:
            result = resp.json()
            status = result.get("status", "Unknown")
            if status in ["Succeeded", "Completed"]:
                return result
            elif status in ["Failed", "Cancelled"]:
                raise Exception(f"Operation {status}: {result}")
        time.sleep(5)
    raise TimeoutError(f"Operation timed out after {timeout}s")

# --- Validate connection ---
try:
    ws_info = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}",
        headers=get_auth_headers()
    ).json()
    print(f"Connected to workspace: {ws_info.get('displayName', 'Unknown')}")
    print(f"   Workspace ID: {WORKSPACE_ID}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("   Make sure WORKSPACE_ID is correct and you have Contributor access.")

## 2. Create Lakehouses (Bronze, Silver, Gold)

In [ ]:
print("=" * 60)
print("STEP 2: Creating Lakehouses")
print("=" * 60)

lakehouse_ids = {}

for lh_name in LAKEHOUSES:
    print(f"\n  Creating {lh_name}...")
    result = fabric_post("items", {
        "displayName": lh_name,
        "type": "Lakehouse"
    })

    if result.get("status") == "already_exists":
        print(f"    Skipped — {lh_name} already exists")
    else:
        print(f"    Created {lh_name}")

# Brief pause for propagation, then retrieve IDs
time.sleep(3)
items = fabric_get("items?type=Lakehouse")
for item in items.get("value", []):
    if item["displayName"] in LAKEHOUSES:
        lakehouse_ids[item["displayName"]] = item["id"]
        print(f"    {item['displayName']} -> {item['id']}")

print(f"\nLakehouses ready: {len(lakehouse_ids)}/{len(LAKEHOUSES)}")

## 3. Create Warehouse

In [ ]:
print("=" * 60)
print("STEP 3: Creating Warehouse")
print("=" * 60)

print(f"\n  Creating {WAREHOUSE_NAME}...")
result = fabric_post("items", {
    "displayName": WAREHOUSE_NAME,
    "type": "Warehouse"
})

if result.get("status") == "already_exists":
    print(f"    Skipped — {WAREHOUSE_NAME} already exists")
else:
    print(f"    Created {WAREHOUSE_NAME}")

# Retrieve warehouse ID
time.sleep(3)
warehouse_id = None
items = fabric_get("items?type=Warehouse")
for item in items.get("value", []):
    if item["displayName"] == WAREHOUSE_NAME:
        warehouse_id = item["id"]
        print(f"    {WAREHOUSE_NAME} -> {warehouse_id}")
        break

print(f"\nWarehouse ready")

## 4. Upload Pipeline Notebooks

In [ ]:
import glob

print("=" * 60)
print("STEP 4: Uploading Pipeline Notebooks")
print("=" * 60)

# Search for .ipynb files in common Fabric notebook locations
NOTEBOOK_SEARCH_PATHS = [
    "/lakehouse/default/Files/notebooks/*.ipynb",
    "./notebooks/*.ipynb",
    "/synfs/notebooks/*.ipynb",
]

notebook_files = []
for pattern in NOTEBOOK_SEARCH_PATHS:
    found = sorted(glob.glob(pattern))
    if found:
        notebook_files = found
        print(f"  Found {len(found)} notebooks at: {pattern}")
        break

if notebook_files:
    uploaded, skipped = 0, 0

    for nb_path in notebook_files:
        nb_filename = os.path.basename(nb_path)
        nb_name = nb_filename.replace(".ipynb", "")

        # Don't re-upload this prereq notebook
        if "prereq" in nb_name:
            continue

        print(f"\n  Uploading {nb_name}...")

        with open(nb_path, "r", encoding="utf-8") as f:
            nb_content = f.read()

        nb_base64 = base64.b64encode(nb_content.encode("utf-8")).decode("utf-8")

        body = {
            "displayName": nb_name,
            "type": "Notebook",
            "definition": {
                "format": "ipynb",
                "parts": [{
                    "path": "artifact.content.ipynb",
                    "payload": nb_base64,
                    "payloadType": "InlineBase64"
                }]
            }
        }

        result = fabric_post("items", body)

        if result.get("status") == "already_exists":
            print(f"    {nb_name} already exists — updating...")
            items = fabric_get("items?type=Notebook")
            existing_id = None
            for item in items.get("value", []):
                if item["displayName"] == nb_name:
                    existing_id = item["id"]
                    break

            if existing_id:
                update_body = {
                    "definition": {
                        "format": "ipynb",
                        "parts": [{
                            "path": "artifact.content.ipynb",
                            "payload": nb_base64,
                            "payloadType": "InlineBase64"
                        }]
                    }
                }
                resp = requests.post(
                    f"{FABRIC_API}/items/{existing_id}/updateDefinition",
                    headers=get_auth_headers(),
                    json=update_body
                )
                if resp.status_code in [200, 202]:
                    print(f"    Updated {nb_name}")
                    uploaded += 1
                else:
                    print(f"    Update failed: {resp.status_code}")
            else:
                skipped += 1
        else:
            print(f"    Created {nb_name}")
            uploaded += 1

    print(f"\nNotebooks uploaded: {uploaded}, skipped: {skipped}")
else:
    print("\n  No notebook files found in search paths.")
    print("  Options:")
    print("    1. Use Fabric Git Integration — notebooks sync automatically")
    print("    2. Upload .ipynb files to Lakehouse Files/notebooks/ first")
    print("    3. Import notebooks via Fabric workspace UI")
    print("\n  If using Git Integration, notebooks are already deployed — skip this step.")

## 5. Lakehouse-Notebook Mapping Reference

In [ ]:
print("=" * 60)
print("STEP 5: Lakehouse-Notebook Mapping")
print("=" * 60)

NOTEBOOK_LAKEHOUSE_MAP = {
    # Bronze
    "nb_01_bronze_ingest":          "lh_bronze_insurance",
    # Silver (default LH = silver; reads bronze via shortcut/cross-LH)
    "nb_02_silver_customers":       "lh_silver_insurance",
    "nb_03_silver_agents":          "lh_silver_insurance",
    "nb_04_silver_policies":        "lh_silver_insurance",
    "nb_05_silver_coverages":       "lh_silver_insurance",
    "nb_06_silver_premiums":        "lh_silver_insurance",
    "nb_07_silver_claims":          "lh_silver_insurance",
    "nb_08_silver_claim_payments":  "lh_silver_insurance",
    # Gold (default LH = gold; reads silver via shortcut/cross-LH)
    "nb_09_gold_claims_summary":    "lh_gold_insurance",
    "nb_10_gold_premium_revenue":   "lh_gold_insurance",
    "nb_11_gold_customer_360":      "lh_gold_insurance",
    "nb_12_gold_kpi_metrics":       "lh_gold_insurance",
}

print("\n  After deployment, set the Default Lakehouse for each notebook:")
print("  " + "-" * 58)
print(f"  {'Notebook':<40} {'Default Lakehouse'}")
print("  " + "-" * 58)
for nb, lh in NOTEBOOK_LAKEHOUSE_MAP.items():
    lh_id = lakehouse_ids.get(lh, "???")
    short_id = lh_id[:8] + "..." if len(lh_id) > 8 else lh_id
    print(f"  {nb:<40} {lh} ({short_id})")

print("\n  How to set in UI: Open notebook -> Explorer pane -> Add Lakehouse -> Existing -> Select")
print("  With Git Integration, lakehouse bindings are in notebook metadata automatically.")

print(f"\n  Lakehouse IDs:")
for name, lid in lakehouse_ids.items():
    print(f"    {name}: {lid}")
if warehouse_id:
    print(f"    {WAREHOUSE_NAME}: {warehouse_id}")

## 6. (Optional) Execute All Notebooks in Order

In [ ]:
print("=" * 60)
print("STEP 6: Execute Notebooks")
print("=" * 60)

if not EXECUTE_AFTER_DEPLOY:
    print("\n  Skipping execution (EXECUTE_AFTER_DEPLOY = False)")
    print("  Set EXECUTE_AFTER_DEPLOY = True in cell 0 to enable.")
    print("\n  Manual execution order:")
    for i, nb in enumerate(NOTEBOOK_EXECUTION_ORDER, 1):
        layer = "Bronze" if "bronze" in nb else "Silver" if "silver" in nb else "Gold" if "gold" in nb else "Setup"
        print(f"    {i:2d}. [{layer:6s}] {nb}")
else:
    print("\n  Starting notebook execution pipeline...")
    print("  Using mssparkutils.notebook.run() for in-workspace execution.")

    results = []

    for nb_name in NOTEBOOK_EXECUTION_ORDER:
        print(f"\n  Running {nb_name}...")
        start_time = time.time()

        try:
            mssparkutils.notebook.run(nb_name, timeout_seconds=600)
            elapsed = time.time() - start_time
            print(f"    Completed ({elapsed:.0f}s)")
            results.append((nb_name, "SUCCESS", elapsed))
        except Exception as e:
            elapsed = time.time() - start_time
            print(f"    FAILED ({elapsed:.0f}s): {str(e)[:200]}")
            results.append((nb_name, "FAILED", elapsed))
            print("\n  Stopping pipeline due to failure.")
            break

    # Summary
    print("\n" + "=" * 60)
    print("EXECUTION SUMMARY")
    print("=" * 60)
    total_time = 0
    for nb_name, status, elapsed in results:
        icon = "[OK]" if status == "SUCCESS" else "[FAIL]"
        print(f"  {icon:6s} {nb_name:<40} {elapsed:.0f}s")
        total_time += elapsed

    success_count = sum(1 for _, s, _ in results if s == "SUCCESS")
    print(f"\n  Succeeded: {success_count}/{len(NOTEBOOK_EXECUTION_ORDER)}")
    print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")

## 7. Deployment Summary

In [ ]:
print("=" * 60)
print("DEPLOYMENT COMPLETE")
print("=" * 60)

print(f"""
  Artifacts Created:
  -----------------------------------------------
  Lakehouses:    {len(lakehouse_ids)} created
                 - lh_bronze_insurance (raw ingestion)
                 - lh_silver_insurance (cleansed/conformed)
                 - lh_gold_insurance   (business aggregates)

  Warehouse:     1 created
                 - {WAREHOUSE_NAME} (SQL analytics)

  Notebooks:     {len(NOTEBOOK_EXECUTION_ORDER)} pipeline notebooks

  -----------------------------------------------
  Next Steps:
  -----------------------------------------------
  1. Set Default Lakehouse for each notebook (see Step 5)
  2. Run nb_00 first (generates mock CSVs to Bronze Lakehouse)
  3. Run nb_01 -> nb_02..nb_08 -> nb_09..nb_12
     OR set EXECUTE_AFTER_DEPLOY = True and re-run Step 6
  4. Create Warehouse shortcuts to Gold Lakehouse tables
  5. Run warehouse/sample_queries.sql against {WAREHOUSE_NAME}
  6. Create Power BI semantic model (sm_insurance)
     using DirectLake mode against lh_gold_insurance tables:
     - gold_claims_summary
     - gold_premium_revenue
     - gold_customer_360
     - gold_kpi_metrics
  7. Build Power BI report (rpt_insurance_dashboard)

  -----------------------------------------------
  Workspace URL:
  https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}
""")

print("  Artifact IDs (save for reference):")
for name, lid in lakehouse_ids.items():
    print(f"    {name}: {lid}")
if warehouse_id:
    print(f"    {WAREHOUSE_NAME}: {warehouse_id}")